# GPIO Example 3 — Alternate functions and remapping

**Target:** STM32F103C8T6 · GPIOA / GPIOB  
**Focus:** Hand a pin from GPIO to a peripheral and reason about pin conflicts.

## Feature classification

| GPIO role | HAL mode | Example | Important constraint |
|---|---|---|---|
| Peripheral output | `GPIO_MODE_AF_PP` | TIM PWM, UART TX, SPI SCK/MOSI | Peripheral drives output; pin needs correct default/remapped route |
| Peripheral open-drain output | `GPIO_MODE_AF_OD` | I2C SCL/SDA | Pull-ups required; use correct voltage and rise time |
| Peripheral input | `GPIO_MODE_INPUT` or AF input as generated | UART RX, timer capture, SPI MISO | Check generated CubeMX configuration and pin route |
| GPIO output | `GPIO_MODE_OUTPUT_PP` | Software-controlled pin | Cannot be used simultaneously by a peripheral |

Unlike newer STM32 parts, STM32F1 does not have a per-pin `GPIOx_AFR` alternate-function selector. The peripheral signal is routed through fixed pins and optional AFIO remap fields. Enable AFIO when changing remaps. Disabling JTAG while retaining SWD can free PA15, PB3, and PB4 for GPIO/peripherals.

## Example — TIM2_CH1 on PA0

The repository project `STM32F103C8T6_TIM_PWM_Output` routes TIM2_CH1–CH3 to PA0–PA2. Default TIM2 mapping uses PA0/PA1/PA2/PA3 for CH1–CH4. Configure PA0 as alternate-function push-pull and configure/start TIM2 PWM. Do not configure PA0 as a regular output or ADC1_IN0 at the same time. A partial/full remap changes available pins; consult the STM32F1 datasheet and RM0008 AFIO map.

```c
GPIO_InitTypeDef gpio = {0};
__HAL_RCC_GPIOA_CLK_ENABLE();
gpio.Pin = GPIO_PIN_0;
gpio.Mode = GPIO_MODE_AF_PP;
gpio.Pull = GPIO_NOPULL;
gpio.Speed = GPIO_SPEED_FREQ_HIGH;
HAL_GPIO_Init(GPIOA, &gpio);

// Then initialize TIM2_CH1 and call:
// HAL_TIM_PWM_Start(&htim2, TIM_CHANNEL_1);
```


## Common conflicts to classify before wiring

- PA13 / PA14 are SWDIO / SWCLK while debug is enabled. Keep them reserved for SWD unless intentionally changing debug configuration.
- PA11 / PA12 are USB D− / D+ on USB-capable boards and also have alternate peripheral functions.
- PA8–PA11 are default TIM1_CH1–CH4; some overlap with other functions. TIM1 complementary outputs use PB13–PB15 for CH1N–CH3N on the default mapping.
- PB6–PB9 are default I2C1 (remappable) and TIM4 channels. They cannot serve both functions concurrently.
- PC13–PC15 have electrical/drive constraints and oscillator-related functions; check the board schematic and datasheet.

## Configuration checklist

1. Confirm the chosen pin is bonded on the LQFP48 package.
2. Check the default peripheral mapping and any required AFIO remap in the datasheet/RM0008.
3. Resolve conflicts with SWD, USB, oscillator pins, and other enabled peripherals.
4. Set the STM32F1 GPIO mode, pull, and output speed; verify external pull-ups and logic levels.
5. Confirm the CubeMX pinout and generated MSP initialization agree with the board wiring.